In [ ]:
%pip install --upgrade pandas matplotlib torch scikit-learn joblib pandas numpy torch

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
import re
import joblib

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

In [ ]:
def basic_preprocess(text: str) -> str:
    """
    Lowercase, remove emails/URLs/mentions/non-alphanumeric chars, and collapse whitespace.
    """
    text = text.lower()
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'[^0-9a-z\s]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def load_and_preprocess_data(file_path: str) -> pd.DataFrame:
    """
    Load CSV, check columns, drop missing, and preprocess statements.
    """
    df = pd.read_csv(file_path)
    if 'statement' not in df.columns or 'status' not in df.columns:
        raise ValueError("Expected 'statement' and 'status' columns.")
    df = df.dropna(subset=['statement', 'status']).reset_index(drop=True)
    df['statement'] = df['statement'].apply(basic_preprocess)
    return df


In [ ]:
def get_metrics(model, X_test, y_test):
    """
    Return a dict of evaluation metrics for a given model.
    """
    y_pred = model.predict(X_test)
    if isinstance(y_pred, np.ndarray) and y_pred.ndim > 1:
        y_pred = np.argmax(y_pred, axis=1)
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'f1': f1_score(y_test, y_pred, average='macro', zero_division=0)
    }

In [ ]:
def vectorize_texts_tfidf(train_texts: list, test_texts: list):
    """
    Fit TF-IDF on train_texts, transform both train and test.
    """
    vec = TfidfVectorizer()
    X_train = vec.fit_transform(train_texts)
    X_test  = vec.transform(test_texts)
    return X_train, X_test, vec

In [ ]:
def train_logistic_regression(X_train, y_train):
    """
    Grid-search LogisticRegression on TF-IDF features; save best model.
    """
    clf = LogisticRegression(
        penalty='l2',
        class_weight='balanced',
        solver='lbfgs',
        max_iter=500,
        random_state=RANDOM_SEED
    )
    param_grid = {'C': [0.01, 0.1, 1, 10]}
    gs = GridSearchCV(
        clf,
        param_grid,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=0,     # no per-fit logging
        refit=True
    )
    gs.fit(X_train, y_train)
    print(f"Best LogisticRegression C: {gs.best_params_['C']}")
    joblib.dump(gs.best_estimator_, 'best_logreg_model.pkl')
    return gs.best_estimator_

In [ ]:
def train_random_forest(X_train, y_train):
    """
    Grid-search RandomForestClassifier; save best model.
    """
    clf = RandomForestClassifier(
        class_weight='balanced',
        random_state=RANDOM_SEED
    )
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20]
    }
    gs = GridSearchCV(
        clf,
        param_grid,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=0,
        refit=True
    )
    gs.fit(X_train, y_train)
    print(f"Best RandomForest params: {gs.best_params_}")
    joblib.dump(gs.best_estimator_, 'best_rf_model.pkl')
    return gs.best_estimator_

In [ ]:
def evaluate_model(name: str, model, X_test, y_test, label_encoder=None):
    """
    Predict on X_test, compute and print accuracy / precision / recall / F1 and confusion matrix.
    """
    y_pred = model.predict(X_test)
    if isinstance(y_pred, np.ndarray) and y_pred.ndim > 1:
        y_pred = np.argmax(y_pred, axis=1)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='macro', zero_division=0)
    print(f"{name} – Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")

    cm = confusion_matrix(y_test, y_pred)
    if label_encoder is not None:
        labels = label_encoder.classes_
        cm_df = pd.DataFrame(cm, index=labels, columns=labels)
        print("Confusion Matrix:")
        print(cm_df)
    else:
        print("Confusion Matrix:")
        print(cm)

In [ ]:
if __name__ == "__main__":
    # --- 1. Load & preprocess
    data_path = "../data/Combined Data.csv"
    df = load_and_preprocess_data(data_path)

    # --- 2. Label encode & split
    le = LabelEncoder()
    df['label'] = le.fit_transform(df['status'])
    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        stratify=df['label'],
        random_state=RANDOM_SEED
    )

    # --- 3. Vectorize
    X_train, X_test, tfidf_vec = vectorize_texts_tfidf(
        train_df['statement'].tolist(),
        test_df ['statement'].tolist()
    )
    y_train = train_df['label'].values
    y_test  = test_df ['label'].values

    # --- 4. Train & save Logistic Regression
    logreg_model = train_logistic_regression(X_train, y_train)
    print("\nLogistic Regression Evaluation:")
    evaluate_model("Logistic Regression", logreg_model, X_test, y_test, label_encoder=le)

    # --- 5. Train & save Random Forest
    rf_model = train_random_forest(X_train, y_train)
    print("\nRandom Forest Evaluation:")
    evaluate_model("Random Forest", rf_model, X_test, y_test, label_encoder=le)

In [ ]:
# Compare and visualize both models
metrics_logreg = get_metrics(logreg_model, X_test, y_test)
metrics_rf     = get_metrics(rf_model,   X_test, y_test)
comp_df = pd.DataFrame(
    [metrics_logreg, metrics_rf],
    index=["Logistic Regression", "Random Forest"]
).T

# Plot bar chart of metric comparisons
comp_df.plot(kind='bar', figsize=(8, 6))
plt.title("Model Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()